In [1]:
from dotenv import load_dotenv

status = load_dotenv(dotenv_path='../../.env')
print(status)

True


## Local MCP server

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [3]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [4]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)
agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='cd7341eb-1a78-4520-9d18-8503425a0e15'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "what is langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'71fbf2c3-2545-4583-83ba-7ee223369be6': 'EjQKMgERTTIP/n+lba9qVp/IxHO6NxpzLv6a4piZE9gxeLNd/MP8Y/agKPeaIWX22Q8L6cKx'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f4113-bd1d-7ed0-a27a-02f791ad102b-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'what is langchain-mcp-adapters library'}, 'id': '71fbf2c3-2545-4583-83ba-7ee223369be6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens': 24, 'total_tokens': 223, 'input_token_details': {'cache_read

## Online MCP

In [20]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [21]:
agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    tools=tools,
)

In [22]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='74b781a8-7518-4ea0-b58a-a58fdffa428e'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "America/New_York"}'}, '__gemini_function_call_thought_signatures__': {'ba5d1c8d-d5cf-4b80-bc1a-fd5a8b4694c4': 'EjQKMgERTTIPpoiLvnS/G2TQZZ3Q5gPmow0iQ1RDLinuHCg1ZFWSYm5sbYZDVeK6dzAlPM/s'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f410c-73d1-7770-ac9e-46a4845003c0-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'ba5d1c8d-d5cf-4b80-bc1a-fd5a8b4694c4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284, 'output_tokens': 22, 'total_tokens': 306, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content=[{'type': 'text